# Create enrichment tables for database

This notebook is intended for use within a Databricks DLT Pipeline and creates Delta Live tables in the enrichment schema. The notebook includes the following steps:

1. **Create or Refresh '_cleaned' Delta Live Tables**: Create the table if it does not exist, else refresh with streaming data. Simple transformations and cleaning is done here.
2. **Create or Refresh '_upserted' Delta Live Tables**: Create the table if it does not exist, else refresh with streaming data. Cleaned data is streamed to an scd-type1 streaming table.
3. **Create or Refresh '_history' Delta Live Tables**: Create the table if it does not exist, else refresh with streaming data. Cleaned data is streamed to an scd-type2 streaming table.
5. **Create or Refresh '_dim' Materialized Views**: Create the view if it does not exist, else refresh with streaming data. These are 'final' dimension tables produced from joining tables '_upserted', '_history' and '_dim' tables.


In [0]:
CREATE OR REFRESH STREAMING TABLE silver.ga_event_cleaned
AS
SELECT
  row.dimensions[0] AS country,
  row.dimensions[1] AS countryId,
  row.dimensions[2] as date_,
  to_date(row.dimensions[2],'yyyyMMdd') AS dateProcess,
  CAST(row.metrics[0] AS INT) AS activeUsers,
  CAST(row.metrics[1] AS INT) AS screenPageViews,
  CAST(row.metrics[2] AS INT) AS scrolledUsers,
  row.metrics[3] AS sessions,
  row.metrics[4] AS sessionsPerUser,
  row.metrics[5] AS userEngagementDuration,
  processing_time,file_path
FROM stream(dataflatform_dev.bronze.ga_event)
LATERAL VIEW explode(
  from_json(
    content.json,
    'STRUCT<headers:STRUCT<dimensions:ARRAY<STRING>,metrics:ARRAY<STRING>>,rows:ARRAY<STRUCT<dimensions:ARRAY<STRING>,metrics:ARRAY<STRING>>>>'
  ).rows
) AS row
;

In [0]:
CREATE OR REFRESH STREAMING TABLE silver.ga_event_upserted
COMMENT "Streaming table containing latest invoice database."
TBLPROPERTIES("table.layer"="silver", "table.type"="transformation as scd-type1");
CREATE FLOW silver_ga_event_upserted_flow
AS AUTO CDC INTO silver.ga_event_upserted
FROM STREAM(silver.ga_event_cleaned)
KEYS (countryId,date_)
SEQUENCE BY processing_time
COLUMNS * EXCEPT (processing_time, file_path)
STORED AS SCD TYPE 1;

In [0]:
CREATE OR REFRESH STREAMING TABLE silver.ga_event_history
COMMENT "Streaming table containing latest invoice database."
TBLPROPERTIES("table.layer"="silver", "table.type"="transformation as scd-type1");
CREATE FLOW silver_ga_event_history_flow
AS AUTO CDC INTO silver.ga_event_history
FROM STREAM(silver.ga_event_cleaned)
KEYS (countryId,date_)
SEQUENCE BY processing_time
COLUMNS * EXCEPT (processing_time, file_path)
STORED AS SCD TYPE 2
TRACK HISTORY ON * EXCEPT (processing_time);

In [0]:
CREATE OR REFRESH STREAMING TABLE gold.ga_event_page_cleaned
AS
SELECT
  row.dimensions[0] AS date,
  row.dimensions[1] AS pagePath,
  row.dimensions[2] AS country,
  row.metrics[0] AS screenPageViews,
  processing_time,file_path
FROM stream(dataflatform_dev.bronze.ga_event_page)
LATERAL VIEW explode(
  from_json(
    content.json,
    'STRUCT<headers:STRUCT<dimensions:ARRAY<STRING>,metrics:ARRAY<STRING>>,rows:ARRAY<STRUCT<dimensions:ARRAY<STRING>,metrics:ARRAY<STRING>>>>'
  ).rows
) AS row
;